### LightGBM + Integrasi Optuna

Load Data 

In [ ]:
import pandas as pd
import numpy as np

print("Loading pre-saved standardized datasets (train/val only)...")

# Load the standardized split datasets (train & validation only)
train_data = pd.read_csv("../dataset_splitted/train_split_scaled.csv")
val_data = pd.read_csv("../dataset_splitted/val_split_scaled.csv")

print("Loaded standardized datasets:")
print("  - train_split_scaled.csv:", train_data.shape)
print("  - val_split_scaled.csv:", val_data.shape)

# Separate features and target
target_col = "category"
X_train = train_data.drop(columns=[target_col]).values.astype(np.float32)
y_train = train_data[target_col].values

X_val = val_data.drop(columns=[target_col]).values.astype(np.float32)
y_val = val_data[target_col].values

# Summary
n_classes = len(np.unique(np.concatenate([y_train, y_val])))
print("\nData ready for Optuna search:")
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}, y_val shape: {y_val.shape}")
print(f"Total classes (from train+val): {n_classes}")
print(f"Data type: {X_train.dtype}")

### Optuna Search (LightGBM)

In [ ]:
from time import perf_counter
import numpy as np
import lightgbm as lgb
import optuna
from optuna.integration import LightGBMPruningCallback
import matplotlib.pyplot as plt
import json
import os
from datetime import datetime, timezone, timedelta

# Create directory for saving hyperparameters
save_dir = "./saved_hyperparams_baseline"
os.makedirs(save_dir, exist_ok=True)

# Define WIB timezone (GMT+7)
WIB = timezone(timedelta(hours=7))

# Prepare LightGBM datasets from pre-loaded arrays (X_train, y_train, X_val, y_val)
lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train)

# Determine number of classes from train+val
n_classes = int(len(np.unique(np.concatenate([y_train, y_val]))))

# Initialize trial logs
trial_logs = []

# Define Optuna objective function
def objective(trial: optuna.trial.Trial) -> float:
    params = {
        # Default
        # 'data_sample_strategy': 'goss', 
        # 'enable_bundle': True,  # EFB 
        
        # Static
        'objective': 'multiclass',
        'num_class': n_classes,
        'metric': 'multi_logloss',
        'feature_pre_filter': False, 
        'verbosity': -1,
        'seed': 42,  # Reproducibility
    }

    # Hyperparameter search space
    params['learning_rate'] = trial.suggest_float('learning_rate', 0.01, 0.2)
    params['num_leaves'] = trial.suggest_int('num_leaves', 64, 512, step=32)
    params['max_depth'] = trial.suggest_int('max_depth', 4, 16, step=2)
    params['min_data_in_leaf'] = trial.suggest_int('min_data_in_leaf', 100, 1500, step=100)
    params['feature_fraction'] = trial.suggest_categorical('feature_fraction', [0.7, 0.8, 0.9])
    params['lambda_l1'] = trial.suggest_categorical('lambda_l1', [0.01, 0.1, 1, 3, 7])
    params['lambda_l2'] = trial.suggest_categorical('lambda_l2', [0.01, 0.1, 1, 3, 7])
    
    # Number of boosting rounds
    num_boost_round = trial.suggest_int('num_boost_round', 100, 600, step=100)

    # Fixed early stopping configuration
    callbacks = [
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        LightGBMPruningCallback(trial, 'multi_logloss', valid_name='valid'),
    ]

    gbm = lgb.train(
        params=params,
        train_set=lgb_train,
        valid_sets=[lgb_val],
        valid_names=['valid'],
        num_boost_round=num_boost_round,
        callbacks=callbacks,
    )

    # Log trial information with WIB timezone
    trial_info = {
        'trial_number': trial.number,
        'params': trial.params,
        'value': float(gbm.best_score['valid']['multi_logloss']),
        'best_iteration': gbm.best_iteration,
        'datetime': datetime.now(WIB).isoformat()
    }
    trial_logs.append(trial_info)
    
    # Save individual trial log
    trial_file = os.path.join(save_dir, f"trial_{trial.number:03d}.json")
    with open(trial_file, 'w') as f:
        json.dump(trial_info, f, indent=2)
    
    print(f"Trial {trial.number}: {float(gbm.best_score['valid']['multi_logloss']):.6f}")

    return float(gbm.best_score['valid']['multi_logloss'])

# Create and run Optuna study with TPESampler
print("Starting Optuna hyperparameter search with TPESampler...")
study_start = perf_counter()

# Configure TPESampler and MedianPruner
sampler = optuna.samplers.TPESampler(seed=42, n_startup_trials=5)
pruner = optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=50, interval_steps=1)

study = optuna.create_study(
    direction='minimize', 
    study_name='lgbm-multiclass-tpe', 
    sampler=sampler,
    pruner=pruner
)
study.optimize(objective, n_trials=15, show_progress_bar=False)
study_end = perf_counter()

print("\nOptuna search completed")
print(f"Total trials: {len(study.trials)}")
print(f"Best validation multi_logloss: {study.best_value:.6f}")
print("Best hyperparameters:")
for param_name, param_value in study.best_params.items():
    print(f"  {param_name}: {param_value}")
print(f"Search duration: {study_end - study_start:.2f} seconds")

# Save complete study results with WIB timezone
study_results = {
    'study_name': 'lgbm-multiclass-tpe',
    'best_value': study.best_value,
    'best_params': study.best_params,
    'best_trial': study.best_trial.number,
    'n_trials': len(study.trials),
    'search_duration': study_end - study_start,
    'datetime': datetime.now(WIB).isoformat(),
    'trials': trial_logs
}

# Save complete results
results_file = os.path.join(save_dir, f"optuna_study_results_{datetime.now(WIB).strftime('%Y%m%d_%H%M%S')}.json")
with open(results_file, 'w') as f:
    json.dump(study_results, f, indent=2)

print(f"\nResults saved to: {results_file}")
print(f"Individual trials saved in: {save_dir}")

# Visualize hyperparameter importance
print("\nGenerating hyperparameter importance visualization...")
try:
    fig = optuna.visualization.plot_param_importances(study)
    fig.update_layout(
        title="Hyperparameter Importance",
        width=800,
        height=600
    )
    fig.show()
except Exception as e:
    print(f"Error generating importance plot: {e}")
    
    # Jika Error: Calculate and display importance as text
    try:
        importance = optuna.importance.get_param_importances(study)
        print("\nHyperparameter Importance (text format):")
        for param, imp in sorted(importance.items(), key=lambda x: x[1], reverse=True):
            print(f"  {param}: {imp:.4f}")
    except Exception as e2:
        print(f"Error calculating importance: {e2}")